# 07 · Gün İçi Stratejiler: ORB ve Alternatifleri
## Bölüm 1 — 15 dakikalık Opening Range Breakout (ORB)

Bu not defteri önceki çalışmalardan (01–06) **tamamen ayrı** bir parkurdur: gün içinde açılıp **aynı gün kapanan** stratejiler. Opsiyon kuralları (21–45 gün vade) burada geçerli değildir; burada yalnızca dayanak varlık (ETF / hisse) üzerinde çalışılır.

**Strateji (temel hali):**
1. Seansın ilk 15 dakikası (09:30–09:45 ET, kış saatinde TSİ 17:30–17:45) **açılış aralığıdır** (OR). Yüksek ve düşüğü kaydedilir.
2. Yön: aralığın kapanışı açılışından yukarıdaysa yalnızca **uzun**, aşağıdaysa yalnızca **kısa** (Zarattini ve ark., 2024). Alternatif: ilk kırılan taraf.
3. Giriş: aralık yükseğine (kısa için düşüğüne) stop emri.
4. Stop: 14 günlük ATR'nin %10'u (makale); alternatifler: aralığın karşı ucu ya da ortası.
5. Çıkış: stop çalışmazsa gün sonunda.

**Kanıt ne diyor?** Zarattini, Barbon ve Aziz (2024), 2016–2023'te göreli hacmi yüksek hisselerde ("Stocks in Play") aralık süresini karşılaştırıyor:

| Aralık | Toplam getiri | Yıllık | Sharpe | İsabet | Maks. düşüş |
|---|---|---|---|---|---|
| 5 dk | %1.637 | %41,6 | 2,81 | %48,4 | %12 |
| **15 dk** | **%272** | **%17,4** | **1,43** | **%44,7** | **%11** |
| 30 dk | %21 | %2,3 | 0,21 | %42,4 | %35 |
| 60 dk | %39 | %4,1 | 0,40 | %42,3 | %21 |

Yani 15 dk, 5 dk'nın çok gerisinde ama 30/60 dk'dan belirgin şekilde iyi. Bu sonuçlar **göreli hacim filtresiyle** ve 1 dakikalık veriyle elde edildi. Bağımsız bir tekrar (5 dk ORB, endeks CFD'leri, 2015–2026) brüt kenarın gerçek olduğunu ama **maliyetten sonra sıfıra yakın** kaldığını buldu.

**Bu not defteri neyi test eder, neyi etmez?**
- ✅ Kuralın mekaniği, maliyet duyarlılığı, **rastgele yönlü girişe karşı** üstünlüğü, aralık süresi ve stop türüne duyarlılık, hangi gün tiplerinde çalıştığı.
- ❌ Makalenin tekrarı değil: veri yfinance'ten, **5 dakikalık mumlar, yalnızca son ~60 gün**. Bu da sembol başına ~40–60 işlem demek; istatistiksel kanıt için az.

**Kullanım:** Ayarları gözden geçir, sonra **Çalışma zamanı > Tümünü çalıştır** (Ctrl+F9). API anahtarı gerekmez.

> ⚠️ Öğrenme amaçlıdır, yatırım tavsiyesi değildir.

In [ ]:
#@title ⚙️ Ayarlar
SEMBOLLER = "QQQ, SPY, IWM, NVDA, TSLA" #@param {type:"string"}
#@markdown Virgülle ayır. Likit ETF'ler ve büyük hisseler önerilir.
OR_DAKIKA = 15 #@param [5, 15, 30, 60] {type:"raw"}
YON_KURALI = "aralık yönü (makale)" #@param ["aralık yönü (makale)", "ilk kırılan taraf"]
TETIK = "anlık kırılım (stop emri)" #@param ["anlık kırılım (stop emri)", "5 dk kapanış teyidi"]
#@markdown Anlık: fiyat aralığa değdiği an girilir. Teyit: 5 dk mum aralığın dışında kapanınca bir sonraki mumun açılışında girilir.
STOP_TURU = "ATR yüzdesi (makale)" #@param ["ATR yüzdesi (makale)", "aralığın karşı ucu", "aralığın ortası"]
ATR_YUZDE = 10 #@param {type:"number"}
#@markdown Yalnızca "ATR yüzdesi" stop türünde kullanılır. Makale: 14 günlük ATR'nin %10'u.
HEDEF_R = 0 #@param {type:"number"}
#@markdown 0 = hedef yok, gün sonunda çık (makale). Örn. 2 = riskin 2 katında kâr al.
SON_GIRIS = "15:30" #@param {type:"string"}
#@markdown ET saati. Bu saatten sonra yeni giriş yapılmaz.
AYNI_MUM = "kötümser" #@param ["kötümser", "iyimser"]
#@markdown 5 dk mumun içinde stop mu önce geldi, hedef mi (ya da giriş mi) bilinmez. Kötümser: stop önce. İyimser: yalnızca kesinse stop. İkisi arasındaki fark, 5 dk verinin cevaplayamadığı kısımdır.
#@markdown ---
#@markdown **Maliyet**
KOMISYON_HISSE = 0.0035 #@param {type:"number"}
#@markdown Hisse başına dolar, her yön için (makale: IBKR Pro Tiered 0,0035 $).
KAYMA_BPS = 1 #@param {type:"number"}
#@markdown Her yön için kayma, baz puan (1 bps = %0,01). Likit ETF'lerde 0,5–1; tek hisselerde 2–5 daha gerçekçi.
#@markdown ---
RASTGELE_TEKRAR = 300 #@param {type:"integer"}
#@markdown Rastgele yönlü kıyasın tekrar sayısı.
TOHUM = 42

## 1 · Kurulum ve yardımcılar
Sayılar Türkçe biçimde gösterilir (1.234,56; %12,5), tarihler GG/AA/YYYY.

In [ ]:
import subprocess, sys
try:
    import yfinance as yf
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "yfinance"], check=True)
    import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

def tr_sayi(x, ond=2):
    """1234.5 -> '1.234,50'"""
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return "–"
    s = f"{abs(x):,.{ond}f}".replace(",", "X").replace(".", ",").replace("X", ".")
    return ("-" if x < 0 else "") + s

def tr_yuzde(x, ond=1):
    """0.125 -> '%12,5' ; -0.05 -> '-%5,0'"""
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return "–"
    return ("-" if x < 0 else "") + "%" + tr_sayi(abs(x) * 100, ond)

def tr_tarih(d):
    return pd.Timestamp(d).strftime("%d/%m/%Y")

def dk(saat):
    """'15:30' -> 930 (gece yarısından itibaren dakika)"""
    s, d = saat.split(":")
    return int(s) * 60 + int(d)

ACILIS_DK = dk("09:30")
assert OR_DAKIKA % 5 == 0, "OR_DAKIKA 5'in katı olmalı (veri 5 dakikalık)."
semboller = [s.strip().upper() for s in SEMBOLLER.split(",") if s.strip()]
print("Semboller:", ", ".join(semboller))

## 2 · Veri
- **5 dakikalık mumlar:** yfinance son ~60 günü veriyor. Seans dışı (pre/post) hariç.
- **Günlük mumlar:** 14 günlük ATR için. ATR **bir önceki güne kadar** hesaplanır; o günün verisi kullanılmaz (geleceği görme hatası olmasın).
- **Göreli hacim:** açılış aralığındaki hacim ÷ önceki 14 günün aynı penceredeki ortalaması. Makaledeki "Stocks in Play" ölçüsü budur. İlk 14 gün için hesaplanamaz.

In [ ]:
def veri_indir(sembol):
    t = yf.Ticker(sembol)
    m5 = t.history(period="59d", interval="5m", prepost=False, auto_adjust=False)
    gun = t.history(period="1y", interval="1d", auto_adjust=False)
    if m5.empty or gun.empty:
        raise ValueError(f"{sembol}: veri gelmedi")
    m5.index = m5.index.tz_convert("America/New_York")
    m5 = m5[["Open", "High", "Low", "Close", "Volume"]]
    m5 = m5[(m5.index.hour * 60 + m5.index.minute >= ACILIS_DK) &
            (m5.index.hour * 60 + m5.index.minute < dk("16:00"))]

    gun.index = pd.to_datetime(gun.index.tz_convert("America/New_York").date)
    onceki_kapanis = gun["Close"].shift(1)
    tr_ = pd.concat([gun["High"] - gun["Low"],
                     (gun["High"] - onceki_kapanis).abs(),
                     (gun["Low"] - onceki_kapanis).abs()], axis=1).max(axis=1)
    gun["ATR14"] = tr_.rolling(14).mean().shift(1)      # yalnızca dünkü bilgi
    gun["OncekiKapanis"] = onceki_kapanis
    return m5, gun

def gunlere_bol(m5, gun):
    """Her işlem günü için numpy dizileri hazırlar."""
    gunler = []
    for tarih, g in m5.groupby(m5.index.date):
        tarih = pd.Timestamp(tarih)
        if tarih not in gun.index or np.isnan(gun.at[tarih, "ATR14"]):
            continue
        dakika = (g.index.hour * 60 + g.index.minute).values
        if dakika[0] != ACILIS_DK:
            continue
        gunler.append(dict(
            tarih=tarih, dk=dakika,
            o=g["Open"].values, h=g["High"].values, l=g["Low"].values,
            c=g["Close"].values, v=g["Volume"].values,
            atr=gun.at[tarih, "ATR14"], onceki_kapanis=gun.at[tarih, "OncekiKapanis"]))
    return gunler

def goreli_hacim_ekle(gunler, or_dk):
    k = or_dk // 5
    or_hacim = [g["v"][:k].sum() for g in gunler]
    for i, g in enumerate(gunler):
        gecmis = or_hacim[max(0, i - 14):i]
        g["goreli_hacim"] = or_hacim[i] / np.mean(gecmis) if len(gecmis) == 14 and np.mean(gecmis) > 0 else np.nan

VERI = {}
for s in semboller:
    try:
        m5, gun = veri_indir(s)
        VERI[s] = gunlere_bol(m5, gun)
        print(f"{s}: {len(VERI[s])} gün, {tr_tarih(VERI[s][0]['tarih'])} – {tr_tarih(VERI[s][-1]['tarih'])}")
    except Exception as e:
        print(f"{s}: atlandı ({e})")

## 3 · Simülasyon motoru
Her gün için tek işlem. **5 dakikalık mumla test etmenin sınırı:** bir mumun içinde fiyatın hangi sırayla hareket ettiğini bilemeyiz. Bu en çok **giriş mumunda** önemli: anlık kırılımda giriş mumun ortasında olur ve mumun dibi girişten önce mi sonra mı geldi, bilinmez. Makaledeki stop (ATR'nin %10'u) çoğu zaman tek bir 5 dk mumun boyundan **küçüktür**, yani bu belirsizlik sonucu doğrudan belirler.

İki varsayım:
- **Kötümser:** mumda stopa değildiyse stop olmuş sayılır; aynı mumda hem stop hem hedef varsa stop önce gelmiştir.
- **İyimser:** giriş mumunda yalnızca kapanış stopun ötesindeyse stop sayılır; stop ve hedef aynı mumdaysa hedef önce gelmiştir.

Gerçek sonuç bu ikisinin arasındadır (sahte veride 1 dk "gerçek" yolla doğrulandı: her iki stop türünde de gerçek sonuç bandın içinde kaldı). Bant geniş ve işaret değiştiriyorsa 5 dk veri bu soruyu cevaplayamıyor demektir, 1 dk veri gerekir. Her iki varsayımda ortak kural: fiyat stopun ötesinde açılırsa (boşluk) dolum açılış fiyatından olur.

**Gizli iyimserlik:** motor, giriş ve stop emirlerinin **tam seviyeden** dolduğunu varsayar. Gerçekte fiyat seviyeyi atlayarak geçer ve dolum biraz kötü olur. Sahte veri testinde bu tek başına kenarsız bir stratejiyi kârlı gösterdi. `KAYMA_BPS` bu farkı karşılamak içindir; stop dar oldukça (ATR %10) kaymanın R'deki payı büyür, o yüzden düşük tutma.

Sonuçlar **R** cinsindendir: 1 R = girişle stop arasındaki mesafe, yani işlem başına alınan risk. Makale her işlemde sermayenin %1'ini riske ettiği için 1 R ≈ sermayenin %1'i.

**Maliyetin R karşılığı:** maliyet hisse başına sabit dolar gibidir, ama stop daraldıkça R'ye oranı büyür. ATR'nin %10'u gibi dar bir stopta maliyet R'nin önemli bir kısmını yiyebilir.

In [ ]:
def gun_simule(g, or_dk, yon_kurali, tetik, stop_turu, atr_yuzde, hedef_r, son_giris_dk,
               komisyon, kayma_bps, ayni_mum="kötümser", yon_zorla=None):
    k = or_dk // 5
    o, h, l, c, d_ = g["o"], g["h"], g["l"], g["c"], g["dk"]
    if len(o) <= k:
        return None
    orh, orl = h[:k].max(), l[:k].min()
    oro, orc = o[0], c[k - 1]
    atr = g["atr"]
    temel = dict(tarih=g["tarih"], or_genislik_atr=(orh - orl) / atr,
                 bosluk_atr=(oro - g["onceki_kapanis"]) / atr,
                 goreli_hacim=g.get("goreli_hacim", np.nan))

    # --- yön ---
    if yon_kurali.startswith("aralık"):
        if abs(orc - oro) < 1e-4 * oro:          # doji: işlem yok
            return None
        yon = 1 if orc > oro else -1
    else:                                         # ilk kırılan taraf
        yon = None
        for j in range(k, len(o)):
            yukari, asagi = h[j] > orh, l[j] < orl
            if yukari and asagi:
                return None                       # aynı mumda iki taraf: belirsiz
            if yukari or asagi:
                yon = 1 if yukari else -1
                break
        if yon is None:
            return None
    if yon_zorla is not None:
        yon = yon_zorla
    seviye = orh if yon == 1 else orl

    # --- giriş ---
    giris = gj = None
    for j in range(k, len(o)):
        if d_[j] > son_giris_dk:
            break
        if tetik.startswith("anlık"):
            if (yon == 1 and h[j] >= seviye) or (yon == -1 and l[j] <= seviye):
                giris = max(seviye, o[j]) if yon == 1 else min(seviye, o[j])
                gj, giris_mumu_ici = j, True
                break
        else:
            if (yon == 1 and c[j] > seviye) or (yon == -1 and c[j] < seviye):
                if j + 1 < len(o):
                    giris, gj, giris_mumu_ici = o[j + 1], j + 1, False
                break
    if giris is None:
        return None

    # --- stop ve hedef ---
    if stop_turu.startswith("ATR"):
        stop = giris - yon * atr * atr_yuzde / 100
    elif stop_turu.startswith("aralığın karşı"):
        stop = orl if yon == 1 else orh
    else:
        stop = (orh + orl) / 2
    risk = (giris - stop) * yon
    if risk <= 0:
        return None
    hedef = giris + yon * hedef_r * risk if hedef_r > 0 else None

    # --- çıkış ---
    cikis, neden, cj = c[-1], "gün sonu", len(o) - 1
    kotumser = (ayni_mum == "kötümser")
    for i in range(gj, len(o)):
        ilk = (i == gj and giris_mumu_ici)
        if not ilk:
            if (yon == 1 and o[i] <= stop) or (yon == -1 and o[i] >= stop):
                cikis, neden, cj = o[i], "stop (boşluk)", i; break
            if hedef is not None and ((yon == 1 and o[i] >= hedef) or (yon == -1 and o[i] <= hedef)):
                cikis, neden, cj = o[i], "hedef", i; break
        stop_degdi = (yon == 1 and l[i] <= stop) or (yon == -1 and h[i] >= stop)
        hedef_degdi = hedef is not None and ((yon == 1 and h[i] >= hedef) or (yon == -1 and l[i] <= hedef))
        # Giriş mumunda stopa değmek belirsizdir (dip girişten önce olmuş olabilir);
        # kapanış stopun ötesindeyse ise kesindir. Hedefe değmek girişten sonra olmak zorundadır.
        stop_kesin = stop_degdi and (not ilk or (yon == 1 and c[i] <= stop) or (yon == -1 and c[i] >= stop))
        if kotumser:
            if stop_degdi:
                cikis, neden, cj = stop, "stop", i; break
            if hedef_degdi:
                cikis, neden, cj = hedef, "hedef", i; break
        else:
            if hedef_degdi:
                cikis, neden, cj = hedef, "hedef", i; break
            if stop_kesin:
                cikis, neden, cj = stop, "stop", i; break

    brut = (cikis - giris) * yon
    maliyet = 2 * komisyon + kayma_bps / 1e4 * (giris + cikis)
    return dict(temel, yon=yon, giris=giris, stop=stop, cikis=cikis, neden=neden,
                giris_saati=d_[gj], cikis_saati=d_[cj], risk=risk,
                maliyet_R=maliyet / risk, R_brut=brut / risk, R_net=(brut - maliyet) / risk,
                getiri_brut=brut / giris)

def calistir(or_dk=None, yon_kurali=None, tetik=None, stop_turu=None, hedef_r=None,
             yon_uretici=None, semboller_=None, ayni_mum=None):
    p = dict(or_dk=or_dk or OR_DAKIKA, yon_kurali=yon_kurali or YON_KURALI,
             tetik=tetik or TETIK, stop_turu=stop_turu or STOP_TURU, atr_yuzde=ATR_YUZDE,
             hedef_r=HEDEF_R if hedef_r is None else hedef_r, son_giris_dk=dk(SON_GIRIS),
             komisyon=KOMISYON_HISSE, kayma_bps=KAYMA_BPS, ayni_mum=ayni_mum or AYNI_MUM)
    satirlar = []
    for s in (semboller_ or VERI):
        goreli_hacim_ekle(VERI[s], p["or_dk"])
        for g in VERI[s]:
            yz = yon_uretici() if yon_uretici else None
            r = gun_simule(g, yon_zorla=yz, **p)
            if r:
                r["sembol"] = s
                satirlar.append(r)
    return pd.DataFrame(satirlar)

def ozet(df):
    if df.empty:
        return pd.Series(dtype=float)
    r = df["R_net"]
    kum = r.cumsum()
    kazanc, kayip = r[r > 0].sum(), -r[r < 0].sum()
    return pd.Series({
        "işlem": len(r),
        "isabet": (r > 0).mean(),
        "ort. R brüt": df["R_brut"].mean(),
        "ort. maliyet R": df["maliyet_R"].mean(),
        "ort. R net": r.mean(),
        "toplam R net": r.sum(),
        "kâr faktörü": kazanc / kayip if kayip > 0 else np.nan,
        "maks. düşüş R": (kum - kum.cummax()).min(),
        "t-ist.": r.mean() / (r.std(ddof=1) / np.sqrt(len(r))) if len(r) > 2 else np.nan,
    })

def ozet_tablosu(df, grup="sembol"):
    t = df.groupby(grup).apply(ozet)
    t.loc["TÜMÜ"] = ozet(df)
    return t

def tr_tablo(t):
    """Özet tablosunu Türkçe biçimli metne çevirir."""
    b = t.copy().astype(object)
    for col in t.columns:
        for idx in t.index:
            x = t.at[idx, col]
            if col == "işlem":
                b.at[idx, col] = tr_sayi(x, 0)
            elif col == "isabet":
                b.at[idx, col] = tr_yuzde(x)
            else:
                b.at[idx, col] = tr_sayi(x, 2)
    return b

## 4 · Ana sonuç: seçtiğin ayarlarla 15 dk ORB
**Nasıl okunur?**
- **ort. R net > 0** olmadan strateji para kazandırmaz. Brüt ile net arasındaki fark maliyetin payıdır.
- **İsabet** ORB'de düşük olur (%40–50); kenar, kazananların kaybedenlerden büyük olmasından gelir. Kâr faktörüne bak.
- **t-ist.** kabaca 2'nin altındaysa sonuç şansla ayırt edilemez. 60 günlük veride bu neredeyse her zaman böyle olacaktır.

In [ ]:
SONUC = calistir()
if SONUC.empty:
    print("Hiç işlem çıkmadı. Ayarları kontrol et.")
else:
    display(Markdown(f"**{OR_DAKIKA} dk ORB · {YON_KURALI} · {TETIK} · stop: {STOP_TURU}"
                     + (f" ({tr_sayi(ATR_YUZDE,0)}% ATR)" if STOP_TURU.startswith("ATR") else "")
                     + f" · hedef: {'yok' if HEDEF_R == 0 else tr_sayi(HEDEF_R,1) + ' R'}**"))
    display(tr_tablo(ozet_tablosu(SONUC)))
    diger = "iyimser" if AYNI_MUM == "kötümser" else "kötümser"
    o_d = ozet(calistir(ayni_mum=diger))
    o_s = ozet(SONUC)
    print(f"\nBelirsizlik bandı (TÜMÜ, ort. R net): {AYNI_MUM} {tr_sayi(o_s['ort. R net'], 3)} · "
          f"{diger} {tr_sayi(o_d['ort. R net'], 3)}")
    if np.sign(o_s['ort. R net']) != np.sign(o_d['ort. R net']):
        print("⚠️ İki varsayım farklı işaret veriyor: 5 dk veri bu ayarın kârlı olup olmadığını söyleyemiyor.")
    print("\nÇıkış nedenleri:")
    print((SONUC["neden"].value_counts(normalize=True) * 100).round(1).astype(str).str.replace(".", ",") + " %")

In [ ]:
#@title 📈 Kümülatif R eğrisi (net)
if not SONUC.empty:
    fig, ax = plt.subplots(figsize=(10, 4))
    for s, g in SONUC.groupby("sembol"):
        ax.plot(g["tarih"], g["R_net"].cumsum().values, label=s, lw=1.5)
    ax.axhline(0, color="grey", lw=0.8)
    ax.set_title(f"{OR_DAKIKA} dk ORB · kümülatif net R (1 R ≈ sermayenin %1'i)")
    ax.set_ylabel("R"); ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

## 5 · Asıl soru: ilk aralığın yönü bilgi taşıyor mu?
Aynı günlerde, aynı giriş/stop/çıkış kurallarıyla ama **yazı-tura ile seçilen yönle** işlem yapılır ve bu yüzlerce kez tekrarlanır. Stratejinin net ortalaması rastgele dağılımın sağ kuyruğunda değilse, "kenar" dediğimiz şey yön bilgisinden değil, piyasanın o dönemki genel gidişatından ya da şanstan geliyor olabilir.

**p-değeri:** rastgele tekrarların kaçta kaçı stratejiye eşit ya da daha iyi sonuç verdi. 0,05'in altı anlamlı kabul edilir.

**Neden rastgele yön bile kazanabilir?** Dar stop ile gün sonuna kadar tutmak, yönden bağımsız olarak **dışbükey** bir getiri üretir: çok sayıda küçük (-1 R) kayıp, seyrek ama büyük kazanç. Bu, uzun gamma / uzun straddle profiline benzer ve trendli günlerde para kazandırır, testere günlerinde kaybettirir. Yani ORB'nin kârı iki parçadan oluşur:
- **Yapı katkısı:** rastgele yönün ortalaması. Dönemin ne kadar "trendli" geçtiğini ölçer.
- **Yön katkısı:** strateji − rastgele medyan. İlk aralığın yönünün gerçekten bilgi taşıyıp taşımadığını gösterir.

Makalenin iddiası ikinci parçayla ilgilidir; ilk parça başka bir yapıyla da (örneğin gün içi straddle) elde edilebilir.

In [ ]:
rng = np.random.default_rng(TOHUM)
rastgele_ort = []
for _ in range(RASTGELE_TEKRAR):
    df_r = calistir(yon_uretici=lambda: int(rng.choice([-1, 1])))
    rastgele_ort.append(df_r["R_net"].mean() if not df_r.empty else np.nan)
rastgele_ort = np.array(rastgele_ort)
strateji_ort = SONUC["R_net"].mean()
p = (np.sum(rastgele_ort >= strateji_ort) + 1) / (np.sum(~np.isnan(rastgele_ort)) + 1)

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(rastgele_ort, bins=30, color="#9aa5b1", edgecolor="white")
ax.axvline(strateji_ort, color="#c0392b", lw=2, label="strateji")
ax.set_title("Rastgele yönlü girişlerin ortalama net R dağılımı"); ax.set_xlabel("ortalama net R")
ax.legend(); plt.tight_layout(); plt.show()
print(f"Strateji ort. net R: {tr_sayi(strateji_ort, 3)}")
print(f"Rastgele ort. net R (medyan): {tr_sayi(np.nanmedian(rastgele_ort), 3)}")
print(f"Yön bilgisinin katkısı: {tr_sayi(strateji_ort - np.nanmedian(rastgele_ort), 3)} R / işlem")
print(f"p-değeri: {tr_sayi(p, 3)}")

## 6 · Duyarlılık: aralık süresi × stop türü
Makalede 5 dk > 15 dk > 60 dk > 30 dk sıralaması çıktı. Tablolar iki mum içi varsayımla ayrı ayrı verilir; geniş stoplarda (aralığın karşı ucu) iki tablo birbirine yakın olmalı, dar stopta (ATR %10) uzaklaşır. Bizim verimizde aynı sıralama görülüyor mu? Tek bir ayarın iyi, komşularının kötü çıkması **aşırı uydurma** (overfitting) işaretidir; sağlam bir kenar komşu ayarlarda da kabaca devam eder.

In [ ]:
satir = []
for or_dk in [5, 15, 30, 60]:
    for st in ["ATR yüzdesi (makale)", "aralığın karşı ucu", "aralığın ortası"]:
        o_k = ozet(calistir(or_dk=or_dk, stop_turu=st, ayni_mum="kötümser"))
        o_i = ozet(calistir(or_dk=or_dk, stop_turu=st, ayni_mum="iyimser"))
        satir.append(dict(OR=f"{or_dk} dk", stop=st, işlem=o_k.get("işlem", 0),
                          net_k=o_k.get("ort. R net", np.nan), net_i=o_i.get("ort. R net", np.nan)))
DUY = pd.DataFrame(satir)
for alan, baslik in [("net_k", "Ortalama net R · kötümser"), ("net_i", "Ortalama net R · iyimser"),
                     ("işlem", "İşlem sayısı")]:
    pv = DUY.pivot(index="OR", columns="stop", values=alan).reindex([f"{x} dk" for x in [5, 15, 30, 60]])
    display(Markdown(f"**{baslik}**"))
    display(pv.apply(lambda col: col.map(lambda x: tr_sayi(x, 0 if alan == "işlem" else 3))))

## 7 · Hangi günlerde çalışıyor?
Üç gün özelliği üçe bölünür (düşük / orta / yüksek):
- **Aralık genişliği / ATR:** dar açılış aralıkları daha sağlıklı kırılım verir mi?
- **Açılış boşluğu / ATR:** büyük boşluklu günlerde kırılım devam ediyor mu, sönüyor mu?
- **Göreli hacim:** makalenin "Stocks in Play" fikri. Hacmi alışılmadık yüksek günler daha mı iyi?

Gruplar küçük olacak; bunu kesin sonuç değil, **hipotez üretimi** olarak oku.

In [ ]:
def ucte_bol(df, kolon, etiket):
    d = df.dropna(subset=[kolon]).copy()
    if len(d) < 9:
        print(f"{etiket}: yeterli veri yok"); return
    d["grup"] = pd.qcut(d[kolon].abs() if kolon == "bosluk_atr" else d[kolon], 3,
                        labels=["düşük", "orta", "yüksek"])
    t = d.groupby("grup", observed=True).apply(ozet)[["işlem", "isabet", "ort. R brüt", "ort. R net", "kâr faktörü"]]
    display(Markdown(f"**{etiket}**")); display(tr_tablo(t))

if not SONUC.empty:
    ucte_bol(SONUC, "or_genislik_atr", "Aralık genişliği / ATR")
    ucte_bol(SONUC, "bosluk_atr", "Açılış boşluğu / ATR (mutlak)")
    ucte_bol(SONUC, "goreli_hacim", "Göreli hacim (açılış aralığı)")

In [ ]:
#@title 🔍 Örnek gün grafiği (en büyük kazanç ve en büyük kayıp)
def gun_ciz(satir):
    g = next(x for x in VERI[satir["sembol"]] if x["tarih"] == satir["tarih"])
    k = OR_DAKIKA // 5
    zaman = [f"{m // 60:02d}:{m % 60:02d}" for m in g["dk"]]
    x = np.arange(len(zaman))
    fig, ax = plt.subplots(figsize=(10, 3.8))
    ax.vlines(x, g["l"], g["h"], color="#7f8c8d", lw=1)
    ax.plot(x, g["c"], color="#2c3e50", lw=1.2)
    ax.axhspan(g["l"][:k].min(), g["h"][:k].max(), color="#f1c40f", alpha=0.2, label="açılış aralığı")
    gi = list(g["dk"]).index(satir["giris_saati"]); ci = list(g["dk"]).index(satir["cikis_saati"])
    ax.scatter([gi], [satir["giris"]], marker="^" if satir["yon"] == 1 else "v", color="#27ae60", s=90, zorder=5, label="giriş")
    ax.scatter([ci], [satir["cikis"]], marker="x", color="#c0392b", s=90, zorder=5, label=f"çıkış ({satir['neden']})")
    ax.axhline(satir["stop"], color="#c0392b", ls="--", lw=0.8, label="stop")
    ax.set_xticks(x[::6]); ax.set_xticklabels(zaman[::6])
    ax.set_title(f"${satir['sembol']} · {tr_tarih(satir['tarih'])} · {tr_sayi(satir['R_net'], 2)} R net")
    ax.legend(loc="best", fontsize=8); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

if not SONUC.empty:
    gun_ciz(SONUC.loc[SONUC["R_net"].idxmax()])
    gun_ciz(SONUC.loc[SONUC["R_net"].idxmin()])

## 8 · X gönderisi
Aşağıdaki **KOPYALAMAK İÇİN DÜZ METİN** bölümünden kopyala; $ işaretleri X'te cashtag olarak çalışır.

In [ ]:
if not SONUC.empty:
    o_ = ozet(SONUC)
    bas, son = SONUC["tarih"].min(), SONUC["tarih"].max()
    cashtag = " ".join(f"${s}" for s in sorted(SONUC["sembol"].unique()))
    gonderi = (
        f"{OR_DAKIKA} dk açılış aralığı kırılımı (ORB) testi\n"
        f"{cashtag} · {tr_tarih(bas)}–{tr_tarih(son)} · 5 dk veri\n\n"
        f"İşlem: {tr_sayi(o_['işlem'], 0)} · İsabet: {tr_yuzde(o_['isabet'])}\n"
        f"Ort. brüt: {tr_sayi(o_['ort. R brüt'], 2)} R · Maliyet sonrası: {tr_sayi(o_['ort. R net'], 2)} R\n"
        f"Rastgele yönle kıyas p={tr_sayi(p, 2)}\n\n"
        f"Ders: kenar maliyete ve stop genişliğine çok duyarlı. "
        f"{'Rastgele girişten ayırt edilemiyor.' if p >= 0.05 else 'Rastgele girişten anlamlı biçimde iyi; ama veri kısa.'}\n\n"
        f"Öğrenme amaçlıdır, yatırım tavsiyesi değildir."
    )
    print("KOPYALAMAK İÇİN DÜZ METİN\n" + "-" * 40)
    print(gonderi)
    print("-" * 40, f"\n{len(gonderi)} karakter")

## 9 · Sınırlar ve sonraki bölümler
**Bu testin zayıf noktaları**
- **60 gün** tek bir piyasa rejimidir. Makalenin 8 yıllık örnekleminde bile zayıf yıllar var.
- **5 dk mum** içi sıra bilinmiyor; motor kötümser varsayım yapıyor, bu da sonuçları biraz aşağı çeker.
- Göreli hacim filtresi ETF'lerde makaledeki gibi çalışmaz; makalenin evreni haber alan tekil hisselerdi.
- Kayma ve komisyon varsayımdır; gerçek dolumla karşılaştırılmadı.

**Yol haritası (07 parkuru)**
1. ✅ 15 dk ORB: mekanik, maliyet, rastgele kıyas, duyarlılık (bu bölüm)
2. Stocks in Play: göreli hacim taramasıyla günlük aday listesi ve ORB'nin bu listede testi
3. Noise Area ("Beat the Market", SPY): gürültü bandı kırılımı, :00/:30 kontrolü, VWAP iz süren stop
4. VWAP trend: makale ve bağımsız tekrar farkının incelenmesi
5. Daha uzun veri: 1 dakikalık, birkaç yıllık geçmiş için ücretsiz/ucuz kaynakların araştırılması

> Öğrenme amaçlıdır, yatırım tavsiyesi değildir.